In [1]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../data/Nassau_Candy_Distributor.csv')
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Ship Date'] = pd.to_datetime(df['Ship Date'], dayfirst=True)

print(f"Raw data loaded: {df.shape[0]} rows, {df.shape[1]} columns")

Raw data loaded: 10194 rows, 18 columns


### Snapshot Before Cleaning

In [3]:
snapshot_before = {
    "total_rows": len(df),
    "null_values": int(df.isnull().sum().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "zero_sales_rows": int((df['Sales'] <= 0).sum()),
    "negative_profit_rows": int((df['Gross Profit'] < 0).sum()),
    "zero_units_rows": int((df['Units'] <= 0).sum()),
    "negative_cost_rows": int((df['Cost'] <= 0).sum())
}

print("BEFORE CLEANING:")
for k, v in snapshot_before.items():
    print(f"  {k}: {v}")

BEFORE CLEANING:
  total_rows: 10194
  null_values: 0
  duplicate_rows: 0
  zero_sales_rows: 0
  negative_profit_rows: 0
  zero_units_rows: 0
  negative_cost_rows: 0


In [4]:
cleaning_log = {
    "issues_found": {},
    "actions_taken": {},
    "rows_removed": {},
    "columns_dropped": {},
    "notes": {}
}

### 1. Check & Drop Nulls

In [5]:
null_counts = df.isnull().sum()
null_cols = null_counts[null_counts > 0]

if len(null_cols) == 0:
    cleaning_log["issues_found"]["null_values"] = "None found"
    cleaning_log["actions_taken"]["null_values"] = "No action needed"
    print("No null values found")
else:
    print(f"Nulls found:\n{null_cols}")
    df.dropna(inplace=True)
    cleaning_log["issues_found"]["null_values"] = null_cols.to_dict()
    cleaning_log["actions_taken"]["null_values"] = "Rows with nulls dropped"

No null values found


### 2. Check & Drop Duplicates

In [6]:
dupe_count = df.duplicated().sum()

if dupe_count == 0:
    cleaning_log["issues_found"]["duplicate_rows"] = "None found"
    cleaning_log["actions_taken"]["duplicate_rows"] = "No action needed"
    print("No duplicate rows found")
else:
    print(f"{dupe_count} duplicate rows found — dropping them")
    df.drop_duplicates(inplace=True)
    cleaning_log["issues_found"]["duplicate_rows"] = int(dupe_count)
    cleaning_log["actions_taken"]["duplicate_rows"] = f"{dupe_count} duplicate rows dropped"

cleaning_log["rows_removed"]["duplicates"] = int(dupe_count)

No duplicate rows found


### 3. Check Zero or Negative Sales

In [7]:
zero_sales = df[df['Sales'] <= 0]

if len(zero_sales) == 0:
    cleaning_log["issues_found"]["zero_or_negative_sales"] = "None found"
    cleaning_log["actions_taken"]["zero_or_negative_sales"] = "No action needed"
    print("No zero or negative sales rows found")
else:
    print(f"{len(zero_sales)} rows with zero/negative sales found — dropping")
    print(zero_sales[['Product Name', 'Sales', 'Gross Profit', 'Cost']].head())
    df = df[df['Sales'] > 0]
    cleaning_log["issues_found"]["zero_or_negative_sales"] = int(len(zero_sales))
    cleaning_log["actions_taken"]["zero_or_negative_sales"] = (
        "Dropped — zero sales rows carry no analytical value "
        "and would distort margin calculations"
    )

cleaning_log["rows_removed"]["zero_or_negative_sales"] = int(len(zero_sales))

No zero or negative sales rows found


### 4. Check Negative Profit

In [8]:
neg_profit = df[df['Gross Profit'] < 0]

if len(neg_profit) == 0:
    cleaning_log["issues_found"]["negative_profit_rows"] = "None found"
    cleaning_log["actions_taken"]["negative_profit_rows"] = "No action needed"
    print("No negative profit rows found")
else:
    print(f"{len(neg_profit)} negative profit rows found")
    print(neg_profit[['Product Name', 'Sales', 'Cost', 'Gross Profit']].head())
    cleaning_log["issues_found"]["negative_profit_rows"] = int(len(neg_profit))
    cleaning_log["actions_taken"]["negative_profit_rows"] = (
        "Retained for now — negative margins are a real business signal "
        "and should be flagged in analysis, not removed"
    )

No negative profit rows found


### 5. Check Zero Units

In [9]:
zero_units = df[df['Units'] <= 0]

if len(zero_units) == 0:
    cleaning_log["issues_found"]["zero_unit_rows"] = "None found"
    cleaning_log["actions_taken"]["zero_unit_rows"] = "No action needed"
    print("No zero unit rows found")
else:
    print(f"{len(zero_units)} rows with zero units — dropping")
    df = df[df['Units'] > 0]
    cleaning_log["issues_found"]["zero_unit_rows"] = int(len(zero_units))
    cleaning_log["actions_taken"]["zero_unit_rows"] = (
        "Dropped — zero unit rows make profit-per-unit calculation "
        "undefined and would cause division by zero errors"
    )

cleaning_log["rows_removed"]["zero_units"] = int(len(zero_units))

No zero unit rows found


### 6. Flag Ship Date Anomaly

In [10]:
# Ship dates range 2026-2030, years after every order date in the raw data
# This column is unreliable — we flag it and exclude from analysis

cleaning_log["issues_found"]["ship_date_anomaly"] = (
    f"Ship dates range from {df['Ship Date'].min().date()} to "
    f"{df['Ship Date'].max().date()} — postdates every order "
    f"({df['Order Date'].min().date()} to {df['Order Date'].max().date()})"
)
cleaning_log["actions_taken"]["ship_date_anomaly"] = (
    "Ship Date column retained in dataset but excluded from all "
    "time-based analysis. Only Order Date will be used."
)
cleaning_log["columns_dropped"]["Ship Date"] = "Excluded from analysis — dates are anomalous"

print("Ship Date anomaly flagged")
print(f"Order Date range : {df['Order Date'].min().date()} → {df['Order Date'].max().date()}")
print(f"Ship Date range  : {df['Ship Date'].min().date()} → {df['Ship Date'].max().date()}")
print("Action           : Ship Date excluded from all time-based analysis")

Ship Date anomaly flagged
Order Date range : 2024-01-02 → 2025-12-31
Ship Date range  : 2026-06-30 → 2030-06-28
Action           : Ship Date excluded from all time-based analysis


### 7. Add Factory Mapping

In [11]:
# Factory data was provided in project brief
# Adding it as a column for factory-level analysis

factory_map = {
    'Wonka Bar - Nutty Crunch Surprise'  : "Lot's O' Nuts",
    'Wonka Bar - Fudge Mallows'          : "Lot's O' Nuts",
    'Wonka Bar -Scrumdiddlyumptious'     : "Lot's O' Nuts",
    'Wonka Bar - Milk Chocolate'         : "Wicked Choccy's",
    'Wonka Bar - Triple Dazzle Caramel'  : "Wicked Choccy's",
    'Laffy Taffy'                        : 'Sugar Shack',
    'SweeTARTS'                          : 'Sugar Shack',
    'Nerds'                              : 'Sugar Shack',
    'Fun Dip'                            : 'Sugar Shack',
    'Fizzy Lifting Drinks'               : 'Sugar Shack',
    'Everlasting Gobstopper'             : 'Secret Factory',
    'Hair Toffee'                        : 'The Other Factory',
    'Lickable Wallpaper'                 : 'Secret Factory',
    'Wonka Gum'                          : 'Secret Factory',
    'Kazookles'                          : 'The Other Factory'
}

df['Factory'] = df['Product Name'].map(factory_map)

unmapped = df['Factory'].isnull().sum()
cleaning_log["notes"]["factory_mapping"] = (
    f"Factory column added from project brief. "
    f"Unmapped products: {unmapped}"
)

print(f"actory column added")
print(f"Unmapped products: {unmapped}")
print(df[['Product Name', 'Factory']].drop_duplicates().to_string(index=False))

actory column added
Unmapped products: 0
                     Product Name           Factory
       Wonka Bar - Milk Chocolate   Wicked Choccy's
Wonka Bar - Triple Dazzle Caramel   Wicked Choccy's
Wonka Bar - Nutty Crunch Surprise     Lot's O' Nuts
   Wonka Bar -Scrumdiddlyumptious     Lot's O' Nuts
        Wonka Bar - Fudge Mallows     Lot's O' Nuts
                        Wonka Gum    Secret Factory
                        Kazookles The Other Factory
               Lickable Wallpaper    Secret Factory
             Fizzy Lifting Drinks       Sugar Shack
                      Laffy Taffy       Sugar Shack
                        SweeTARTS       Sugar Shack
                            Nerds       Sugar Shack
                      Hair Toffee The Other Factory
           Everlasting Gobstopper    Secret Factory
                          Fun Dip       Sugar Shack


### 8. Restrict Scope to FY2025

In [12]:
# Raw Order Date actually spans two calendar years (2024-01-02 to 2025-12-31),
# not just 2025. Every downstream notebook groups by Month/Quarter only, which
# would silently blend 2024 and 2025 orders into the same "month" bucket.
# Scope is restricted to FY2025 here so the rest of the pipeline is a clean,
# accurate single-year analysis.

rows_before_year_filter = len(df)
year_counts = df['Order Date'].dt.year.value_counts().sort_index()

df = df[df['Order Date'].dt.year == 2025].reset_index(drop=True)

rows_removed_year = rows_before_year_filter - len(df)

cleaning_log["issues_found"]["multi_year_data"] = (
    f"Raw data spans years {year_counts.index.min()}-{year_counts.index.max()}: "
    f"{ {int(k): int(v) for k, v in year_counts.items()} }"
)
cleaning_log["actions_taken"]["scope_to_fy2025"] = (
    f"Filtered to Order Date year 2025 only — {rows_removed_year} rows from 2024 "
    "excluded. Keeps monthly/quarterly trend analysis a clean single-year read "
    "instead of blending two years under the same calendar month."
)
cleaning_log["rows_removed"]["out_of_scope_year"] = int(rows_removed_year)

print(f"Rows by year before filtering : { {int(k): int(v) for k, v in year_counts.items()} }")
print(f"Restricted to FY2025 — removed {rows_removed_year} rows from 2024")
print(f"Remaining rows                : {len(df)}")

Rows by year before filtering : {2024: 4181, 2025: 6013}
Restricted to FY2025 — removed 4181 rows from 2024
Remaining rows                : 6013


### Snapshot After Cleaning

In [13]:
snapshot_after = {
    "total_rows": len(df),
    "null_values": int(df.isnull().sum().sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "zero_sales_rows": int((df['Sales'] <= 0).sum()),
    "negative_profit_rows": int((df['Gross Profit'] < 0).sum()),
    "zero_units_rows": int((df['Units'] <= 0).sum())
}

rows_removed = snapshot_before["total_rows"] - snapshot_after["total_rows"]

print("AFTER CLEANING")
for k, v in snapshot_after.items():
    print(f"  {k}: {v}")

print(f"\n  Total rows removed : {rows_removed}")
print(f"  Data retained      : {snapshot_after['total_rows']:,} rows")

AFTER CLEANING
  total_rows: 6013
  null_values: 0
  duplicate_rows: 0
  zero_sales_rows: 0
  negative_profit_rows: 0
  zero_units_rows: 0

  Total rows removed : 4181
  Data retained      : 6,013 rows


### Save Cleaned Data & Report

In [14]:
# Save cleaned dataframe
cleaned_data_path = '../data/nassau_candy_cleaned.csv'
df.to_csv(cleaned_data_path, index=False)
print(f"Cleaned data saved to: {cleaned_data_path}")

# Build full cleaning report
cleaning_report = {
    "snapshot_before": snapshot_before,
    "snapshot_after": snapshot_after,
    "rows_removed_total": snapshot_before["total_rows"] - snapshot_after["total_rows"],
    "cleaning_log": cleaning_log
}

report_path = '../outputs/reports/data_cleaning_report.json'
os.makedirs(os.path.dirname(report_path), exist_ok=True)

with open(report_path, 'w') as f:
    json.dump(cleaning_report, f, indent=4)

print(f"Cleaning report saved to: {report_path}")

Cleaned data saved to: ../data/nassau_candy_cleaned.csv
Cleaning report saved to: ../outputs/reports/data_cleaning_report.json
